In [37]:
# ============================================================
#   STEP 1 — IMPORT LIBRARIES
# ============================================================

# ── Data Manipulation ────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Utilities ────────────────────────────────────────────────
import warnings
import joblib
import os

warnings.filterwarnings("ignore")

# ── Sklearn: Preprocessing & Splitting ──────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, label_binarize

# ── Sklearn: Models ─────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier   # GradientBoosting = XGBoost equivalent

# ── Sklearn: Evaluation Metrics ─────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

# ── Global Settings ──────────────────────────────────────────
plt.rcParams.update({"figure.dpi": 130, "font.family": "DejaVu Sans"})

PALETTE    = ["#E74C3C", "#F39C12", "#2ECC71"]         # per class colour
MODEL_CLR  = {
    "Decision Tree" : "#3498DB",
    "Random Forest" : "#9B59B6",
    "XGBoost"       : "#E74C3C",
}
CLASS_ORDER = ["High Dementia Risk", "Mild Cognitive Risk", "No Cognitive Risk"]
FEATURES    = ["Word Recall Score",
               "Number Sequencing Time (sec)",
               "Speech Fluency (words/min)"]

OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 55)
print("  ✅  STEP 1 COMPLETE — All Libraries Imported")
print("=" * 55)
print(f"\n  pandas      : {pd.__version__}")
print(f"  numpy       : {np.__version__}")
print(f"  matplotlib  : {plt.matplotlib.__version__}")
print(f"  seaborn     : {sns.__version__}")
import sklearn
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  joblib      : {joblib.__version__}")

  ✅  STEP 1 COMPLETE — All Libraries Imported

  pandas      : 2.3.3
  numpy       : 1.24.3
  matplotlib  : 3.10.5
  seaborn     : 0.13.2
  scikit-learn: 1.6.1
  joblib      : 1.5.1


In [38]:
# ============================================================
#   STEP 2 — LOAD DATASET
# ============================================================

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ── Load CSV ─────────────────────────────────────────────────
DATA_PATH = "dementia_risk_data_v2.csv"

df = pd.read_csv(DATA_PATH)

# ── Basic Info ───────────────────────────────────────────────
print("=" * 55)
print("  ✅  STEP 2 COMPLETE — Dataset Loaded")
print("=" * 55)

print(f"\n  📁 File        : {DATA_PATH}")
print(f"  📐 Shape       : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n  📋 Column Names:")
for col in df.columns:
    print(f"      • {col}  [{df[col].dtype}]")

print(f"\n  🔢 First 5 Rows:")
print(df.head().to_string(index=False))

print(f"\n  📊 Target Column — 'Risk Label' :")
print(df["Risk Label"].value_counts().to_string())
print(f"\n  Total : {len(df)} samples")

  ✅  STEP 2 COMPLETE — Dataset Loaded

  📁 File        : dementia_risk_data_v2.csv
  📐 Shape       : 9999 rows × 4 columns

  📋 Column Names:
      • Word Recall Score  [int64]
      • Number Sequencing Time (sec)  [int64]
      • Speech Fluency (words/min)  [int64]
      • Risk Label  [object]

  🔢 First 5 Rows:
 Word Recall Score  Number Sequencing Time (sec)  Speech Fluency (words/min)          Risk Label
                12                            63                          40 Mild Cognitive Risk
                12                            58                          41 Mild Cognitive Risk
                 3                            54                          33  High Dementia Risk
                17                            33                          56   No Cognitive Risk
                 6                            95                          33  High Dementia Risk

  📊 Target Column — 'Risk Label' :
Risk Label
High Dementia Risk     4759
Mild Cognitive Risk    2879


In [39]:
# ============================================================
#   STEP 3 — DATA CLEANING
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("dementia_risk_data_v2.csv")

FEATURES = ["Word Recall Score",
            "Number Sequencing Time (sec)",
            "Speech Fluency (words/min)"]

print("=" * 55)
print("  STEP 3 — DATA CLEANING")
print("=" * 55)

# ── 3a. Shape & Data Types ───────────────────────────────────
print("\n  [3a] Shape & Data Types")
print(f"       Rows    : {df.shape[0]}")
print(f"       Columns : {df.shape[1]}")
print()
print(df.dtypes.to_string())

# ── 3b. Missing Values ───────────────────────────────────────
print("\n  [3b] Missing Values (per column)")
missing = df.isnull().sum()
print(missing.to_string())
if missing.sum() == 0:
    print("       ✅  No missing values found.")
else:
    print(f"       ⚠️  Total missing: {missing.sum()}")
    df = df.dropna()
    print(f"       Rows after dropna: {len(df)}")

# ── 3c. Duplicate Rows ───────────────────────────────────────
print("\n  [3c] Duplicate Rows")
dupes = df.duplicated().sum()
print(f"       Duplicates found : {dupes}")
df = df.drop_duplicates()
print(f"       Rows after removal: {len(df)}")

# ── 3d. Descriptive Statistics ──────────────────────────────
print("\n  [3d] Descriptive Statistics (Numeric Features)")
print(df[FEATURES].describe().round(2).to_string())

# ── 3e. Outlier Detection (IQR Method) ──────────────────────
print("\n  [3e] Outlier Detection — IQR Method")
print(f"  {'Feature':<35} {'Q1':>6} {'Q3':>6} {'IQR':>6} {'Lower':>8} {'Upper':>8} {'Outliers':>10}")
print("  " + "-" * 83)
for col in FEATURES:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo  = Q1 - 1.5 * IQR
    hi  = Q3 + 1.5 * IQR
    n_out = df[(df[col] < lo) | (df[col] > hi)].shape[0]
    print(f"  {col:<35} {Q1:>6.1f} {Q3:>6.1f} {IQR:>6.1f} {lo:>8.1f} {hi:>8.1f} {n_out:>10}")

# ── 3f. Class Distribution ───────────────────────────────────
print("\n  [3f] Target Class Distribution")
vc = df["Risk Label"].value_counts()
for label, count in vc.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"       {label:<22}: {count:>5}  ({pct:.1f}%)  {bar}")

# ── 3g. Value Ranges ─────────────────────────────────────────
print("\n  [3g] Feature Value Ranges")
for col in FEATURES:
    print(f"       {col:<35}: min={df[col].min():.0f}  max={df[col].max():.0f}  mean={df[col].mean():.2f}")

print("\n" + "=" * 55)
print("  ✅  STEP 3 COMPLETE — Clean DataFrame Ready")
print(f"      Final Shape: {df.shape}")
print("=" * 55)

  STEP 3 — DATA CLEANING

  [3a] Shape & Data Types
       Rows    : 9999
       Columns : 4

Word Recall Score                int64
Number Sequencing Time (sec)     int64
Speech Fluency (words/min)       int64
Risk Label                      object

  [3b] Missing Values (per column)
Word Recall Score               0
Number Sequencing Time (sec)    0
Speech Fluency (words/min)      0
Risk Label                      0
       ✅  No missing values found.

  [3c] Duplicate Rows
       Duplicates found : 3205
       Rows after removal: 6794

  [3d] Descriptive Statistics (Numeric Features)
       Word Recall Score  Number Sequencing Time (sec)  Speech Fluency (words/min)
count            6794.00                       6794.00                     6794.00
mean                9.54                         74.44                       42.31
std                 5.99                         21.64                        7.82
min                 0.00                         30.00                     

In [40]:
# ============================================================
#   STEP 4 — EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 130, "font.family": "DejaVu Sans"})

df = pd.read_csv("dementia_risk_data_v2.csv")

FEATURES    = ["Word Recall Score",
               "Number Sequencing Time (sec)",
               "Speech Fluency (words/min)"]
LABEL_COL   = "Risk Label"
CLASS_ORDER = ["High Dementia Risk", "Mild Cognitive Risk", "No Cognitive Risk"]
PALETTE     = ["#E74C3C", "#F39C12", "#2ECC71"]
OUTPUT_DIR  = "/mnt/user-data/outputs"

counts = df[LABEL_COL].value_counts().reindex(CLASS_ORDER)

# ── Build full EDA figure (4 rows × 3 cols) ──────────────────
fig = plt.figure(figsize=(20, 22))
fig.suptitle("Dementia Risk — Exploratory Data Analysis",
             fontsize=18, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Row 0 : Class distribution bar | Pie | Correlation heatmap
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(CLASS_ORDER, counts, color=PALETTE, edgecolor="white", linewidth=0.8)
ax1.set_title("Class Distribution", fontweight="bold")
ax1.set_ylabel("Count")
ax1.set_xticklabels(CLASS_ORDER, rotation=13, ha="right", fontsize=8)
for b, c in zip(bars, counts):
    ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + 30,
             f"{c}\n({c/len(df)*100:.1f}%)", ha="center", fontsize=8)

ax2 = fig.add_subplot(gs[0, 1])
ax2.pie(counts, labels=CLASS_ORDER, colors=PALETTE, autopct="%1.1f%%",
        startangle=140, textprops={"fontsize": 8})
ax2.set_title("Class Share (%)", fontweight="bold")

ax3 = fig.add_subplot(gs[0, 2])
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", ax=ax3, linewidths=0.5, annot_kws={"size": 9})
ax3.set_title("Feature Correlation", fontweight="bold")
ax3.tick_params(labelsize=7)

# ── Row 1 : Histograms per feature ───────────────────────────
for i, feat in enumerate(FEATURES):
    ax = fig.add_subplot(gs[1, i])
    for j, cls in enumerate(CLASS_ORDER):
        ax.hist(df[df[LABEL_COL] == cls][feat], bins=20,
                alpha=0.65, label=cls, color=PALETTE[j], edgecolor="white")
    ax.set_title(f"Distribution: {feat}", fontweight="bold", fontsize=9)
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel("Count")
    ax.legend(fontsize=6)

# ── Row 2 : Boxplots per feature ─────────────────────────────
for i, feat in enumerate(FEATURES):
    ax = fig.add_subplot(gs[2, i])
    data_by_class = [df[df[LABEL_COL] == cls][feat].values for cls in CLASS_ORDER]
    bp = ax.boxplot(data_by_class, patch_artist=True,
                    medianprops={"color": "black", "linewidth": 2})
    for patch, clr in zip(bp["boxes"], PALETTE):
        patch.set_facecolor(clr)
        patch.set_alpha(0.75)
    ax.set_xticklabels(CLASS_ORDER, rotation=12, fontsize=7)
    ax.set_title(f"Boxplot: {feat}", fontweight="bold", fontsize=9)
    ax.set_ylabel(feat, fontsize=7)

# ── Row 3 : Scatter plots + Mean feature bars ────────────────
ax7 = fig.add_subplot(gs[3, 0])
for j, cls in enumerate(CLASS_ORDER):
    sub = df[df[LABEL_COL] == cls]
    ax7.scatter(sub["Word Recall Score"],
                sub["Number Sequencing Time (sec)"],
                alpha=0.35, s=10, color=PALETTE[j], label=cls)
ax7.set_xlabel("Word Recall Score", fontsize=8)
ax7.set_ylabel("Seq. Time (sec)", fontsize=8)
ax7.set_title("Word Recall vs Seq. Time", fontweight="bold", fontsize=9)
ax7.legend(fontsize=6)

ax8 = fig.add_subplot(gs[3, 1])
for j, cls in enumerate(CLASS_ORDER):
    sub = df[df[LABEL_COL] == cls]
    ax8.scatter(sub["Word Recall Score"],
                sub["Speech Fluency (words/min)"],
                alpha=0.35, s=10, color=PALETTE[j], label=cls)
ax8.set_xlabel("Word Recall Score", fontsize=8)
ax8.set_ylabel("Speech Fluency (words/min)", fontsize=8)
ax8.set_title("Word Recall vs Speech Fluency", fontweight="bold", fontsize=9)
ax8.legend(fontsize=6)

ax9 = fig.add_subplot(gs[3, 2])
means = df.groupby(LABEL_COL)[FEATURES].mean().reindex(CLASS_ORDER)
x = np.arange(len(FEATURES))
w = 0.25
for i, (cls, clr) in enumerate(zip(CLASS_ORDER, PALETTE)):
    ax9.bar(x + i * w, means.loc[cls], w, label=cls, color=clr, alpha=0.85)
ax9.set_xticks(x + w)
ax9.set_xticklabels(["Word Recall", "Seq. Time", "Speech Fluency"], fontsize=7)
ax9.set_title("Mean Feature Values by Class", fontweight="bold")
ax9.legend(fontsize=6)
ax9.set_ylabel("Mean Value")

plt.savefig(f"EDA.png", bbox_inches="tight")
plt.close()

print("=" * 55)
print("  ✅  STEP 4 COMPLETE — EDA")
print("=" * 55)
print(f"\n  Chart saved → {OUTPUT_DIR}/01_EDA.png")
print("\n  Summary Statistics:")
print(df[FEATURES].describe().round(2).to_string())
print("\n  Class Counts:")
print(counts.to_string())

  ✅  STEP 4 COMPLETE — EDA

  Chart saved → /mnt/user-data/outputs/01_EDA.png

  Summary Statistics:
       Word Recall Score  Number Sequencing Time (sec)  Speech Fluency (words/min)
count            9999.00                       9999.00                     9999.00
mean                9.17                         75.54                       41.80
std                 6.06                         21.42                        7.84
min                 0.00                         30.00                       30.00
25%                 4.00                         59.00                       35.00
50%                 9.00                         76.00                       41.00
75%                15.00                         92.00                       48.00
max                20.00                        120.00                       60.00

  Class Counts:
Risk Label
High Dementia Risk     4759
Mild Cognitive Risk    2879
No Cognitive Risk      2361


In [41]:
# ============================================================
#   STEP 5 — LABEL ENCODING & TRAIN / TEST SPLIT
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("dementia_risk_data_v2.csv")

FEATURES  = ["Word Recall Score",
             "Number Sequencing Time (sec)",
             "Speech Fluency (words/min)"]
LABEL_COL = "Risk Label"

print("=" * 55)
print("  STEP 5 — LABEL ENCODING & TRAIN/TEST SPLIT")
print("=" * 55)

# ── 5a. Label Encoding ───────────────────────────────────────
print("\n  [5a] Label Encoding")
le = LabelEncoder()
df["Risk_Encoded"] = le.fit_transform(df[LABEL_COL])

print("\n  Mapping:")
for label, code in zip(le.classes_, range(len(le.classes_))):
    print(f"       {label:<25} →  {code}")

print("\n  Encoded value counts:")
print(df["Risk_Encoded"].value_counts().sort_index().to_string())

# ── 5b. Define X and y ──────────────────────────────────────
X = df[FEATURES]
y = df["Risk_Encoded"]

print(f"\n  [5b] Feature Matrix (X)")
print(f"       Shape   : {X.shape}")
print(f"       Columns : {list(X.columns)}")
print(f"\n  [5b] Target Vector (y)")
print(f"       Shape   : {y.shape}")
print(f"       Unique  : {sorted(y.unique())}")

# ── 5c. Train / Test Split (80 / 20 stratified) ─────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # keeps class proportions equal in both sets
)

print(f"\n  [5c] Train / Test Split  (80% / 20%)")
print(f"       Total   samples : {len(df)}")
print(f"       Training samples: {X_train.shape[0]}  ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"       Testing  samples: {X_test.shape[0]}   ({X_test.shape[0]/len(df)*100:.1f}%)")

print("\n  Class distribution in TRAIN set:")
for code, label in enumerate(le.classes_):
    n = (y_train == code).sum()
    print(f"       {label:<25}: {n}  ({n/len(y_train)*100:.1f}%)")

print("\n  Class distribution in TEST set:")
for code, label in enumerate(le.classes_):
    n = (y_test == code).sum()
    print(f"       {label:<25}: {n}  ({n/len(y_test)*100:.1f}%)")

print("\n" + "=" * 55)
print("  ✅  STEP 5 COMPLETE")
print("  X_train, X_test, y_train, y_test ready.")
print("=" * 55)

  STEP 5 — LABEL ENCODING & TRAIN/TEST SPLIT

  [5a] Label Encoding

  Mapping:
       High Dementia Risk        →  0
       Mild Cognitive Risk       →  1
       No Cognitive Risk         →  2

  Encoded value counts:
Risk_Encoded
0    4759
1    2879
2    2361

  [5b] Feature Matrix (X)
       Shape   : (9999, 3)
       Columns : ['Word Recall Score', 'Number Sequencing Time (sec)', 'Speech Fluency (words/min)']

  [5b] Target Vector (y)
       Shape   : (9999,)
       Unique  : [0, 1, 2]

  [5c] Train / Test Split  (80% / 20%)
       Total   samples : 9999
       Training samples: 7999  (80.0%)
       Testing  samples: 2000   (20.0%)

  Class distribution in TRAIN set:
       High Dementia Risk       : 3807  (47.6%)
       Mild Cognitive Risk      : 2303  (28.8%)
       No Cognitive Risk        : 1889  (23.6%)

  Class distribution in TEST set:
       High Dementia Risk       : 952  (47.6%)
       Mild Cognitive Risk      : 576  (28.8%)
       No Cognitive Risk        : 472  (23.6%)


In [42]:
# ============================================================
#   STEP 6 — MODEL SELECTION & TRAINING
# ============================================================

import pandas as pd
import numpy as np
import time
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
warnings.filterwarnings("ignore")

# ── Load & Prepare Data ──────────────────────────────────────
df = pd.read_csv("dementia_risk_data_v2.csv")
FEATURES = ["Word Recall Score",
            "Number Sequencing Time (sec)",
            "Speech Fluency (words/min)"]

le = LabelEncoder()
y  = le.fit_transform(df["Risk Label"])
X  = df[FEATURES]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("=" * 60)
print("  STEP 6 — MODEL SELECTION & TRAINING")
print("=" * 60)

# ── Model Definitions with Hyperparameters ───────────────────
models = {

    # ── Model 1: Decision Tree ─────────────────────────────
    #    Simple, interpretable tree-based classifier.
    #    max_depth=8     → prevents overfitting
    #    min_samples_split=10 → needs 10+ samples to split a node
    #    min_samples_leaf=5   → each leaf needs 5+ samples
    #    class_weight='balanced' → handles class imbalance
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ),

    # ── Model 2: Random Forest ─────────────────────────────
    #    Ensemble of many decision trees — reduces variance.
    #    n_estimators=200 → 200 trees in the forest
    #    max_depth=12     → deeper than a single tree is OK in ensemble
    #    n_jobs=-1        → uses all CPU cores
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ),

    # ── Model 3: XGBoost (GradientBoosting equivalent) ────
    #    Builds trees sequentially, each correcting prior errors.
    #    learning_rate=0.1 → shrinks each tree's contribution
    #    subsample=0.8     → 80% of rows per tree (prevents overfit)
    "XGBoost": GradientBoostingClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        min_samples_split=5,
        random_state=42
    ),
}

# ── Train Each Model ─────────────────────────────────────────
trained_models = {}
print(f"\n  Training on {X_train.shape[0]} samples "
      f"with {X_train.shape[1]} features ...\n")

for name, model in models.items():
    print(f"  ▶  Training {name} ...", end="  ")
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    trained_models[name] = model
    print(f"✅  Done in {elapsed:.2f}s")

print("\n" + "=" * 60)
print("  ✅  STEP 6 COMPLETE — All 3 Models Trained")
print("=" * 60)

print("\n  Model Hyperparameter Summary:")
print(f"\n  {'Model':<18} Key Hyperparameters")
print("  " + "-" * 58)
print(f"  {'Decision Tree':<18} max_depth=8, min_samples_split=10, min_samples_leaf=5")
print(f"  {'Random Forest':<18} n_estimators=200, max_depth=12, n_jobs=-1")
print(f"  {'XGBoost':<18} n_estimators=200, lr=0.1, subsample=0.8")

  STEP 6 — MODEL SELECTION & TRAINING

  Training on 7999 samples with 3 features ...

  ▶  Training Decision Tree ...  ✅  Done in 0.01s
  ▶  Training Random Forest ...  ✅  Done in 0.57s
  ▶  Training XGBoost ...  ✅  Done in 8.02s

  ✅  STEP 6 COMPLETE — All 3 Models Trained

  Model Hyperparameter Summary:

  Model              Key Hyperparameters
  ----------------------------------------------------------
  Decision Tree      max_depth=8, min_samples_split=10, min_samples_leaf=5
  Random Forest      n_estimators=200, max_depth=12, n_jobs=-1
  XGBoost            n_estimators=200, lr=0.1, subsample=0.8


In [43]:
# ============================================================
#   STEP 7 — MODEL EVALUATION
#            Confusion Matrix + Classification Metrics
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130})

OUTPUT_DIR = "saved_models"

# ── Prepare Data ─────────────────────────────────────────────
df = pd.read_csv(f"dementia_risk_data_v2.csv")
FEATURES = ["Word Recall Score",
            "Number Sequencing Time (sec)",
            "Speech Fluency (words/min)"]

le = LabelEncoder()
y  = le.fit_transform(df["Risk Label"])
X  = df[FEATURES]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

CLASS_NAMES = le.classes_

# ── Train Models ─────────────────────────────────────────────
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_split=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": GradientBoostingClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=42),
}
for m in models.values():
    m.fit(X_train, y_train)

print("=" * 65)
print("  STEP 7 — MODEL EVALUATION")
print("=" * 65)

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Per-Model Metrics ─────────────────────────────────────────
for name, model in models.items():
    y_pred   = model.predict(X_test)
    acc      = accuracy_score(y_test, y_pred)
    prec     = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec      = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1       = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

    results[name] = {
        "Accuracy":  acc,
        "Precision": prec,
        "Recall":    rec,
        "F1-Score":  f1,
        "CV Mean":   cv_scores.mean(),
        "CV Std":    cv_scores.std(),
        "y_pred":    y_pred,
    }

    print(f"\n  {'─'*55}")
    print(f"  MODEL : {name}")
    print(f"  {'─'*55}")
    print(f"  Accuracy    : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall      : {rec:.4f}")
    print(f"  F1-Score    : {f1:.4f}")
    print(f"  CV Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  (5-fold)")

    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# ── Confusion Matrix Plots ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Confusion Matrices — All Models", fontsize=16, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    y_pred = results[name]["y_pred"]
    cm     = confusion_matrix(y_test, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    # Combine count + % into annotation
    annot = np.array([
        [f"{v}\n({p:.1f}%)" for v, p in zip(row_v, row_p)]
        for row_v, row_p in zip(cm, cm_pct)
    ])

    sns.heatmap(
        cm, annot=annot, fmt="", cmap="Blues", ax=ax,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, cbar=True
    )
    ax.set_title(
        f"{name}\n"
        f"Acc: {results[name]['Accuracy']:.4f}  |  "
        f"F1: {results[name]['F1-Score']:.4f}",
        fontweight="bold", fontsize=10
    )
    ax.set_xlabel("Predicted Label", fontsize=9)
    ax.set_ylabel("True Label", fontsize=9)
    ax.tick_params(axis="x", labelsize=7, rotation=15)
    ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_Confusion_Matrices.png", bbox_inches="tight")
plt.close()

print("\n" + "=" * 65)
print("  ✅  STEP 7 COMPLETE — Evaluation Done")
print(f"Confusion_Matrices.png")
print("=" * 65)

# ── Summary Table ─────────────────────────────────────────────
print("\n  METRICS SUMMARY TABLE")
print(f"\n  {'Model':<18} {'Accuracy':>9} {'Precision':>10} "
      f"{'Recall':>8} {'F1-Score':>9} {'CV Mean':>9}")
print("  " + "─" * 60)
for name in results:
    r = results[name]
    print(f"  {name:<18} {r['Accuracy']:>9.4f} {r['Precision']:>10.4f} "
          f"{r['Recall']:>8.4f} {r['F1-Score']:>9.4f} {r['CV Mean']:>9.4f}")

  STEP 7 — MODEL EVALUATION

  ───────────────────────────────────────────────────────
  MODEL : Decision Tree
  ───────────────────────────────────────────────────────
  Accuracy    : 0.9755  (97.55%)
  Precision   : 0.9755
  Recall      : 0.9755
  F1-Score    : 0.9754
  CV Accuracy : 0.9770 ± 0.0021  (5-fold)

  Classification Report:
                     precision    recall  f1-score   support

 High Dementia Risk       0.98      0.99      0.99       952
Mild Cognitive Risk       0.97      0.96      0.96       576
  No Cognitive Risk       0.98      0.96      0.97       472

           accuracy                           0.98      2000
          macro avg       0.97      0.97      0.97      2000
       weighted avg       0.98      0.98      0.98      2000


  ───────────────────────────────────────────────────────
  MODEL : Random Forest
  ───────────────────────────────────────────────────────
  Accuracy    : 0.9760  (97.60%)
  Precision   : 0.9760
  Recall      : 0.9760
  F1-Score 

In [44]:
# ============================================================
#   STEP 8 — MODEL COMPARISON
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130})

OUTPUT_DIR = "saved_models"
MODEL_CLR  = {"Decision Tree": "#3498DB", "Random Forest": "#9B59B6", "XGBoost": "#E74C3C"}

# ── Prepare & Train ──────────────────────────────────────────
df = pd.read_csv(f"dementia_risk_data_v2.csv")
FEATURES = ["Word Recall Score",
            "Number Sequencing Time (sec)",
            "Speech Fluency (words/min)"]
le = LabelEncoder()
y  = le.fit_transform(df["Risk Label"])
X  = df[FEATURES]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_split=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": GradientBoostingClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=42),
}
for m in models.values():
    m.fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, model in models.items():
    yp = model.predict(X_test)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    results[name] = {
        "Accuracy":  accuracy_score(y_test, yp),
        "Precision": precision_score(y_test, yp, average="weighted", zero_division=0),
        "Recall":    recall_score(y_test, yp, average="weighted", zero_division=0),
        "F1-Score":  f1_score(y_test, yp, average="weighted", zero_division=0),
        "CV Mean":   cv_scores.mean(),
        "CV Std":    cv_scores.std(),
    }

METRICS = ["Accuracy", "Precision", "Recall", "F1-Score", "CV Mean"]

print("=" * 65)
print("  STEP 8 — MODEL COMPARISON")
print("=" * 65)
print(f"\n  {'Model':<18} {'Accuracy':>9} {'Precision':>10} "
      f"{'Recall':>8} {'F1-Score':>9} {'CV Mean':>9}")
print("  " + "─" * 60)
for name, r in results.items():
    print(f"  {name:<18} {r['Accuracy']:>9.4f} {r['Precision']:>10.4f} "
          f"{r['Recall']:>8.4f} {r['F1-Score']:>9.4f} {r['CV Mean']:>9.4f}")
best = max(results, key=lambda n: results[n]["F1-Score"])
print(f"\n  🏆  Best Model: {best}  (F1={results[best]['F1-Score']:.4f})")

# ── Figure ───────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 6))
fig.suptitle("Model Comparison Dashboard", fontsize=16, fontweight="bold")

# ── Chart 1: Grouped bar chart ───────────────────────────────
ax1 = fig.add_subplot(1, 3, 1)
x, w = np.arange(len(METRICS)), 0.22
for i, (name, clr) in enumerate(MODEL_CLR.items()):
    vals = [results[name][m] for m in METRICS]
    bars = ax1.bar(x + i * w, vals, w, label=name, color=clr, alpha=0.85, edgecolor="white")
    for b in bars:
        ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.003,
                 f"{b.get_height():.3f}", ha="center", fontsize=6.5)
ax1.set_xticks(x + w)
ax1.set_xticklabels(METRICS, fontsize=9)
ax1.set_ylim(0.70, 1.12)
ax1.set_ylabel("Score")
ax1.set_title("All Metrics Comparison", fontweight="bold")
ax1.legend(fontsize=8)
ax1.axhline(0.95, color="grey", linestyle="--", alpha=0.5, linewidth=1, label="95% line")
ax1.axhline(0.98, color="green", linestyle="--", alpha=0.4, linewidth=1)

# ── Chart 2: Radar chart ─────────────────────────────────────
ax2 = fig.add_axes([0.37, 0.08, 0.28, 0.80], polar=True)
angles = np.linspace(0, 2 * np.pi, len(METRICS), endpoint=False).tolist()
angles += angles[:1]
ax2.set_theta_offset(np.pi / 2)
ax2.set_theta_direction(-1)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(METRICS, fontsize=8)
ax2.set_ylim(0.70, 1.0)
for name, clr in MODEL_CLR.items():
    vals = [results[name][m] for m in METRICS] + [results[name][METRICS[0]]]
    ax2.plot(angles, vals, color=clr, linewidth=2, label=name)
    ax2.fill(angles, vals, color=clr, alpha=0.12)
ax2.set_title("Radar Chart", fontweight="bold", pad=18)
ax2.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)

# ── Chart 3: CV accuracy with error bars ─────────────────────
ax3 = fig.add_axes([0.70, 0.12, 0.27, 0.74])
names   = list(MODEL_CLR.keys())
cv_means = [results[n]["CV Mean"] for n in names]
cv_stds  = [results[n]["CV Std"]  for n in names]
bars = ax3.bar(names, cv_means, color=list(MODEL_CLR.values()),
               alpha=0.85, edgecolor="white", width=0.5)
ax3.errorbar(names, cv_means, yerr=cv_stds,
             fmt="none", color="black", capsize=8, linewidth=2)
for b, m, s in zip(bars, cv_means, cv_stds):
    ax3.text(b.get_x() + b.get_width() / 2, b.get_height() + s + 0.005,
             f"{m:.4f}\n±{s:.4f}", ha="center", fontsize=8, fontweight="bold")
ax3.set_ylim(0.85, 1.04)
ax3.set_title("5-Fold CV Accuracy ± Std", fontweight="bold")
ax3.set_ylabel("CV Accuracy")
ax3.axhline(0.95, color="grey", linestyle="--", alpha=0.5, linewidth=1)
ax3.tick_params(axis="x", labelsize=9)

plt.savefig(f"{OUTPUT_DIR}/03_Model_Comparison.png", bbox_inches="tight")
plt.close()

print(f"\n  ✅  STEP 8 COMPLETE")
print(f"  Chart saved → {OUTPUT_DIR}/03_Model_Comparison.png")

  STEP 8 — MODEL COMPARISON

  Model               Accuracy  Precision   Recall  F1-Score   CV Mean
  ────────────────────────────────────────────────────────────
  Decision Tree         0.9755     0.9755   0.9755    0.9754    0.9770
  Random Forest         0.9760     0.9760   0.9760    0.9760    0.9779
  XGBoost               0.9685     0.9685   0.9685    0.9684    0.9704

  🏆  Best Model: Random Forest  (F1=0.9760)

  ✅  STEP 8 COMPLETE
  Chart saved → saved_models/03_Model_Comparison.png


In [45]:
# ============================================================
#   STEP 9 — FEATURE IMPORTANCE
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130})

OUTPUT_DIR = "saved_models"
MODEL_CLR  = {"Decision Tree": "#3498DB", "Random Forest": "#9B59B6", "XGBoost": "#E74C3C"}

# ── Prepare & Train ──────────────────────────────────────────
df = pd.read_csv(f"dementia_risk_data_v2.csv")
FEATURES = ["Word Recall Score",
            "Number Sequencing Time (sec)",
            "Speech Fluency (words/min)"]
SHORT_F  = ["Word Recall", "Seq. Time (sec)", "Speech Fluency"]

le = LabelEncoder()
y  = le.fit_transform(df["Risk Label"])
X  = df[FEATURES]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_split=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": GradientBoostingClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=42),
}
for m in models.values():
    m.fit(X_train, y_train)

print("=" * 60)
print("  STEP 9 — FEATURE IMPORTANCE")
print("=" * 60)

# ── Console output ────────────────────────────────────────────
for name, model in models.items():
    imp = model.feature_importances_
    print(f"\n  {name}")
    print("  " + "─" * 45)
    for feat, score in sorted(zip(FEATURES, imp), key=lambda x: -x[1]):
        bar = "█" * int(score * 50)
        print(f"  {feat:<38}: {score:.4f}  {bar}")
    print(f"  Most important: {FEATURES[np.argmax(imp)]}")

# ── Feature Importance Plot ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Feature Importance — All Models", fontsize=15, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    imp = model.feature_importances_
    sorted_idx = np.argsort(imp)

    bars = ax.barh(
        np.array(SHORT_F)[sorted_idx],
        imp[sorted_idx],
        color=MODEL_CLR[name], alpha=0.85, edgecolor="white", height=0.5
    )

    for bar in bars:
        ax.text(
            bar.get_width() + 0.005,
            bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.4f}",
            va="center", fontsize=10, fontweight="bold"
        )

    ax.set_title(name, fontweight="bold", fontsize=12)
    ax.set_xlabel("Importance Score", fontsize=9)
    ax.set_xlim(0, max(imp) * 1.30)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_Feature_Importance.png", bbox_inches="tight")
plt.close()

print(f"\n  ✅  STEP 9 COMPLETE")
print(f"  Chart saved → {OUTPUT_DIR}/04_Feature_Importance.png")

# ── Insight ───────────────────────────────────────────────────
print("\n  KEY INSIGHT:")
print("  'Word Recall Score' is the dominant predictor across")
print("  all three models, driving classification decisions.")
print("  Sequencing Time and Speech Fluency provide additional")
print("  discriminating signal especially at class boundaries.")

  STEP 9 — FEATURE IMPORTANCE

  Decision Tree
  ─────────────────────────────────────────────
  Word Recall Score                     : 0.8924  ████████████████████████████████████████████
  Speech Fluency (words/min)            : 0.0982  ████
  Number Sequencing Time (sec)          : 0.0095  
  Most important: Word Recall Score

  Random Forest
  ─────────────────────────────────────────────
  Word Recall Score                     : 0.4836  ████████████████████████
  Speech Fluency (words/min)            : 0.3453  █████████████████
  Number Sequencing Time (sec)          : 0.1711  ████████
  Most important: Word Recall Score

  XGBoost
  ─────────────────────────────────────────────
  Word Recall Score                     : 0.8886  ████████████████████████████████████████████
  Speech Fluency (words/min)            : 0.0871  ████
  Number Sequencing Time (sec)          : 0.0242  █
  Most important: Word Recall Score

  ✅  STEP 9 COMPLETE
  Chart saved → saved_models/04_Feature_Import

In [46]:
# ============================================================
#   STEP 10 — ROC CURVES  (One-vs-Rest, per class per model)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_curve, auc
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130})

OUTPUT_DIR = "saved_models"
MODEL_CLR  = {"Decision Tree": "#3498DB", "Random Forest": "#9B59B6", "XGBoost": "#E74C3C"}
LINE_CLR   = ["#E74C3C", "#F39C12", "#2ECC71"]

# ── Prepare & Train ──────────────────────────────────────────
df = pd.read_csv(f"dementia_risk_data_v2.csv")
FEATURES = ["Word Recall Score",
            "Number Sequencing Time (sec)",
            "Speech Fluency (words/min)"]

le = LabelEncoder()
y  = le.fit_transform(df["Risk Label"])
X  = df[FEATURES]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Binarize y_test for One-vs-Rest ROC
y_test_bin  = label_binarize(y_test, classes=[0, 1, 2])
CLASS_NAMES = le.classes_

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_split=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": GradientBoostingClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=42),
}
for m in models.values():
    m.fit(X_train, y_train)

print("=" * 60)
print("  STEP 10 — ROC CURVES")
print("=" * 60)

# ── Console AUC scores ────────────────────────────────────────
for name, model in models.items():
    y_prob = model.predict_proba(X_test)
    print(f"\n  {name}")
    print("  " + "─" * 45)
    macro_auc = 0
    for i, cls in enumerate(CLASS_NAMES):
        _, _, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc = auc(*roc_curve(y_test_bin[:, i], y_prob[:, i])[:2])
        macro_auc += roc_auc
        print(f"  {cls:<25}: AUC = {roc_auc:.4f}")
    print(f"  {'Macro AUC':<25}: {macro_auc/3:.4f}")

# ── ROC Curve Plots ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ROC Curves (One-vs-Rest) — All Models",
             fontsize=15, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    y_prob = model.predict_proba(X_test)
    macro_auc = 0

    for i, (cls, clr) in enumerate(zip(CLASS_NAMES, LINE_CLR)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc     = auc(fpr, tpr)
        macro_auc  += roc_auc
        ax.plot(fpr, tpr,
                label=f"{cls}\n(AUC = {roc_auc:.3f})",
                color=clr, linewidth=2.5)

    # Diagonal reference line
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.6, label="Random Chance")

    ax.fill_between([0, 1], [0, 1], alpha=0.04, color="gray")
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=8)
    ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=8)
    ax.set_title(
        f"{name}\nMacro AUC = {macro_auc/3:.4f}",
        fontweight="bold", fontsize=11
    )
    ax.legend(fontsize=7, loc="lower right", framealpha=0.85)
    ax.grid(alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_ROC_Curves.png", bbox_inches="tight")
plt.close()

print(f"\n  ✅  STEP 10 COMPLETE")
print(f"  Chart saved → {OUTPUT_DIR}/05_ROC_Curves.png")

print("""
  HOW TO READ ROC CURVES:
  ─────────────────────────────────────────────────
  • AUC (Area Under Curve) closer to 1.0 = better.
  • AUC = 0.5 → random guessing (diagonal line).
  • One-vs-Rest: each class is treated as positive
    vs all others combined.
  ─────────────────────────────────────────────────
""")

  STEP 10 — ROC CURVES

  Decision Tree
  ─────────────────────────────────────────────
  High Dementia Risk       : AUC = 0.9893
  Mild Cognitive Risk      : AUC = 0.9792
  No Cognitive Risk        : AUC = 0.9821
  Macro AUC                : 0.9835

  Random Forest
  ─────────────────────────────────────────────
  High Dementia Risk       : AUC = 0.9891
  Mild Cognitive Risk      : AUC = 0.9851
  No Cognitive Risk        : AUC = 0.9863
  Macro AUC                : 0.9868

  XGBoost
  ─────────────────────────────────────────────
  High Dementia Risk       : AUC = 0.9883
  Mild Cognitive Risk      : AUC = 0.9827
  No Cognitive Risk        : AUC = 0.9845
  Macro AUC                : 0.9852

  ✅  STEP 10 COMPLETE
  Chart saved → saved_models/05_ROC_Curves.png

  HOW TO READ ROC CURVES:
  ─────────────────────────────────────────────────
  • AUC (Area Under Curve) closer to 1.0 = better.
  • AUC = 0.5 → random guessing (diagonal line).
  • One-vs-Rest: each class is treated as positive
  

In [47]:
# ============================================================
#   STEP 11 — MODEL SAVING & LOADING  (with Scaler)
# ============================================================

import pandas as pd
import numpy as np
import joblib
import os
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")

OUTPUT_DIR = "saved_models"

# ── Prepare Data ─────────────────────────────────────────────
df = pd.read_csv(f"dementia_risk_data_v2.csv")
FEATURES = ["Word Recall Score",
            "Number Sequencing Time (sec)",
            "Speech Fluency (words/min)"]

le = LabelEncoder()
y  = le.fit_transform(df["Risk Label"])
X  = df[FEATURES]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Fit Scaler on Training Data ONLY ─────────────────────────
#    Rule: fit on train → transform both train & test
#    Prevents data leakage from test set into scaling
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)   # fit + transform
X_test_sc   = scaler.transform(X_test)        # transform only

print("=" * 60)
print("  STEP 11 — SCALER INFO")
print("=" * 60)
print(f"\n  Scaler Type : StandardScaler")
print(f"  Formula     : z = (x - mean) / std\n")
print(f"  {'Feature':<38} {'Mean':>8} {'Std':>8}")
print("  " + "─" * 56)
for feat, mean, std in zip(FEATURES, scaler.mean_, scaler.scale_):
    print(f"  {feat:<38} {mean:>8.4f} {std:>8.4f}")

print(f"\n  X_train shape (scaled) : {X_train_sc.shape}")
print(f"  X_test  shape (scaled) : {X_test_sc.shape}")

# ── Train Models on Scaled Data ───────────────────────────────
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_split=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": GradientBoostingClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    yp = model.predict(X_test_sc)
    results[name] = {
        "Accuracy": accuracy_score(y_test, yp),
        "F1-Score": f1_score(y_test, yp, average="weighted", zero_division=0),
    }

best_name  = max(results, key=lambda n: results[n]["F1-Score"])
best_model = models[best_name]

# ── Save All Models ───────────────────────────────────────────
print("\n" + "=" * 60)
print("  SAVING MODELS")
print("=" * 60)
print("\n  [a] Individual Models")
for name, model in models.items():
    path = f"{OUTPUT_DIR}/model_{name.replace(' ', '_')}.pkl"
    joblib.dump(model, path)
    size = os.path.getsize(path) / 1024
    print(f"  ✅  {name:<18} → {os.path.basename(path):<35} ({size:.1f} KB)")

print(f"\n  [b] Best Model  →  {best_name}  (F1={results[best_name]['F1-Score']:.4f})")
best_path = f"{OUTPUT_DIR}/best_model_{best_name.replace(' ', '_')}.pkl"
joblib.dump(best_model, best_path)
print(f"  ✅  Saved → {os.path.basename(best_path)}")

# ── Save Scaler ───────────────────────────────────────────────
print(f"\n  [c] Scaler")
scaler_path = f"{OUTPUT_DIR}/scaler.pkl"
joblib.dump(scaler, scaler_path)
size = os.path.getsize(scaler_path) / 1024
print(f"  ✅  StandardScaler saved → scaler.pkl  ({size:.2f} KB)")

# ── Save Label Encoder & Feature Names ───────────────────────
print(f"\n  [d] Label Encoder & Feature Names")
joblib.dump(le,                     f"{OUTPUT_DIR}/label_encoder.pkl")
joblib.dump({"features": FEATURES}, f"{OUTPUT_DIR}/feature_names.pkl")
print(f"  ✅  label_encoder.pkl")
print(f"  ✅  feature_names.pkl")

# ────────────────────────────────────────────────────────────
#   LOADING & PREDICTION DEMO  (using saved scaler)
# ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  LOADING & PREDICTION DEMO")
print("=" * 60)

loaded_model  = joblib.load(best_path)
loaded_scaler = joblib.load(scaler_path)
loaded_le     = joblib.load(f"{OUTPUT_DIR}/label_encoder.pkl")
loaded_feats  = joblib.load(f"{OUTPUT_DIR}/feature_names.pkl")["features"]

print(f"\n  ✅  Loaded model   : {os.path.basename(best_path)}")
print(f"  ✅  Loaded scaler  : scaler.pkl")
print(f"  ✅  Loaded encoder : label_encoder.pkl")

# Verify accuracy is preserved after reload
y_pred_check = loaded_model.predict(loaded_scaler.transform(X_test))
acc_check    = accuracy_score(y_test, y_pred_check)
print(f"\n  Accuracy after reload : {acc_check:.4f}")
print(f"  Matches original      : {round(acc_check, 4) == round(results[best_name]['Accuracy'], 4)}")

# ── Predict 3 new patients (must scale first!) ───────────────
new_patients = pd.DataFrame({
    "Word Recall Score":             [2,  12, 18],
    "Number Sequencing Time (sec)":  [95, 68, 50],
    "Speech Fluency (words/min)":    [33, 44, 54],
})

# ⚠️  IMPORTANT: Always scale new data with the LOADED scaler
new_patients_scaled = loaded_scaler.transform(new_patients)

preds      = loaded_model.predict(new_patients_scaled)
pred_proba = loaded_model.predict_proba(new_patients_scaled)
labels     = loaded_le.inverse_transform(preds)

print("\n  ── Predictions on 3 New Patients ───────────────────")
print(f"  {'Patient':<10} {'Recall':>7} {'SeqTime':>9} "
      f"{'Fluency':>9}  →  Predicted Label               Confidence")
print("  " + "─" * 78)
for i, (label, proba) in enumerate(zip(labels, pred_proba)):
    row  = new_patients.iloc[i]
    conf = proba.max() * 100
    print(f"  Patient {i+1:<4} {row['Word Recall Score']:>7.0f} "
          f"{row['Number Sequencing Time (sec)']:>9.0f} "
          f"{row['Speech Fluency (words/min)']:>9.0f}  →  "
          f"{label:<30} {conf:.2f}%")

print("\n  Probability Breakdown:")
print(f"  {'':12}", end="")
for cn in loaded_le.classes_:
    print(f"  {cn[:22]:<24}", end="")
print()
for i, proba in enumerate(pred_proba):
    print(f"  Patient {i+1:<5}", end="")
    for p in proba:
        print(f"  {p:.4f}{'':18}", end="")
    print()

# ── Final File Tree ───────────────────────────────────────────
print("\n" + "=" * 60)
print("  ✅  STEP 11 COMPLETE")
print("=" * 60)
print(f"\n  Saved files in: {OUTPUT_DIR}/")
print("  ├── model_Decision_Tree.pkl")
print("  ├── model_Random_Forest.pkl")
print("  ├── model_XGBoost.pkl")
print(f"  ├── best_model_{best_name.replace(' ', '_')}.pkl")
print("  ├── scaler.pkl           ← StandardScaler (fit on train)")
print("  ├── label_encoder.pkl")
print("  └── feature_names.pkl")
print("""
  ⚠️  USAGE REMINDER:
  When deploying / predicting on new data always:
    1. Load  scaler.pkl        → loaded_scaler
    2. Load  best_model_*.pkl  → loaded_model
    3. Load  label_encoder.pkl → loaded_le
    4. Scale new data FIRST    → X_new_scaled = loaded_scaler.transform(X_new)
    5. Predict                 → preds = loaded_model.predict(X_new_scaled)
    6. Decode labels           → loaded_le.inverse_transform(preds)
""")

  STEP 11 — SCALER INFO

  Scaler Type : StandardScaler
  Formula     : z = (x - mean) / std

  Feature                                    Mean      Std
  ────────────────────────────────────────────────────────
  Word Recall Score                        9.1828   6.0538
  Number Sequencing Time (sec)            75.5244  21.4040
  Speech Fluency (words/min)              41.8130   7.8479

  X_train shape (scaled) : (7999, 3)
  X_test  shape (scaled) : (2000, 3)

  SAVING MODELS

  [a] Individual Models
  ✅  Decision Tree      → model_Decision_Tree.pkl             (19.8 KB)
  ✅  Random Forest      → model_Random_Forest.pkl             (9468.9 KB)
  ✅  XGBoost            → model_XGBoost.pkl                   (5231.3 KB)

  [b] Best Model  →  Random Forest  (F1=0.9760)
  ✅  Saved → best_model_Random_Forest.pkl

  [c] Scaler
  ✅  StandardScaler saved → scaler.pkl  (0.97 KB)

  [d] Label Encoder & Feature Names
  ✅  label_encoder.pkl
  ✅  feature_names.pkl

  LOADING & PREDICTION DEMO

  ✅  L

In [48]:
# ============================================================
#   STEP 12 — PREDICTION TESTING
#   Using Best Saved Model : Random Forest
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "saved_models"

# ── Load Saved Artefacts ─────────────────────────────────────
scaler        = joblib.load(f"saved_models\scaler.pkl")
label_encoder = joblib.load(f"saved_models\label_encoder.pkl")
model         = joblib.load(f"saved_models/best_model_Random_Forest.pkl")

print("=" * 55)
print("  ✅  Best Model (Random Forest) Loaded")
print("=" * 55)

# ============================================================
#   ➡  ENTER YOUR PATIENT VALUES HERE
#      Add as many rows as you want
#
#   Feature Ranges:
#   Word Recall Score          →  0  to 20
#   Number Sequencing Time     →  30 to 120  (seconds)
#   Speech Fluency             →  30 to 60   (words/min)
# ============================================================

input_data = {
    "Word Recall Score":            [2,  11, 18],
    "Number Sequencing Time (sec)": [95, 68, 50],
    "Speech Fluency (words/min)":   [33, 44, 54],
}

# ── Scale & Predict ──────────────────────────────────────────
df_input   = pd.DataFrame(input_data)
df_scaled  = scaler.transform(df_input)

preds      = model.predict(df_scaled)
pred_proba = model.predict_proba(df_scaled)
labels     = label_encoder.inverse_transform(preds)

# ── Print Results ────────────────────────────────────────────
print(f"\n  {'Patient':<10} {'Recall':>7} {'Seq Time':>10} {'Fluency':>9}  →  {'Predicted Label':<26} {'Confidence':>11}")
print("  " + "─" * 80)

for i, (label, proba) in enumerate(zip(labels, pred_proba)):
    row  = df_input.iloc[i]
    conf = proba.max() * 100
    print(f"  Patient {i+1:<4} {row['Word Recall Score']:>7.0f} "
          f"{row['Number Sequencing Time (sec)']:>10.0f} "
          f"{row['Speech Fluency (words/min)']:>9.0f}  →  "
          f"{label:<26} {conf:>10.2f}%")

print("\n  Probability Breakdown:")
print(f"\n  {'':12}", end="")
for cls in label_encoder.classes_:
    print(f"  {cls:<26}", end="")
print()
print("  " + "─" * 90)
for i, proba in enumerate(pred_proba):
    print(f"  Patient {i+1:<5}", end="")
    for p in proba:
        print(f"  {p:.4f}{'':20}", end="")
    print()

print("\n" + "=" * 55)
print("  ✅  PREDICTION COMPLETE")
print("=" * 55)

  ✅  Best Model (Random Forest) Loaded

  Patient     Recall   Seq Time   Fluency  →  Predicted Label             Confidence
  ────────────────────────────────────────────────────────────────────────────────
  Patient 1          2         95        33  →  High Dementia Risk              98.93%
  Patient 2         11         68        44  →  Mild Cognitive Risk             99.39%
  Patient 3         18         50        54  →  No Cognitive Risk               98.49%

  Probability Breakdown:

                High Dementia Risk          Mild Cognitive Risk         No Cognitive Risk         
  ──────────────────────────────────────────────────────────────────────────────────────────
  Patient 1      0.9893                      0.0067                      0.0040                    
  Patient 2      0.0011                      0.9939                      0.0050                    
  Patient 3      0.0056                      0.0094                      0.9849                    

  ✅  PREDIC

In [49]:
# ============================================================
#   STEP 12 — INTERACTIVE PREDICTION TESTING
#   Enter feature values manually via keyboard
#   Best Model : Random Forest
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = "saved_models"

# ── Load Saved Artefacts ─────────────────────────────────────
scaler        = joblib.load(f"saved_models\scaler.pkl")
label_encoder = joblib.load(f"saved_models\label_encoder.pkl")
model         = joblib.load(f"saved_models/best_model_Random_Forest.pkl")

print("=" * 55)
print("  DEMENTIA RISK PREDICTION SYSTEM")
print("  Model : Random Forest (Best Model)")
print("=" * 55)
print("""
  Feature Input Ranges:
  ┌─────────────────────────────────────┬──────────┐
  │ Feature                             │  Range   │
  ├─────────────────────────────────────┼──────────┤
  │ Word Recall Score                   │  0 – 20  │
  │ Number Sequencing Time (seconds)    │ 30 – 120 │
  │ Speech Fluency (words/min)          │ 30 – 60  │
  └─────────────────────────────────────┴──────────┘
""")

# ── Helper: validated input ───────────────────────────────────
def get_input(prompt, min_val, max_val):
    while True:
        try:
            val = float(input(f"  ➤  {prompt}: "))
            if min_val <= val <= max_val:
                return val
            else:
                print(f"      ⚠️  Please enter a value between {min_val} and {max_val}.\n")
        except ValueError:
            print("      ⚠️  Invalid input. Please enter a number.\n")

# ── Prediction loop ───────────────────────────────────────────
session = 1
while True:
    print(f"\n{'─' * 55}")
    print(f"  Patient {session} — Enter Feature Values")
    print(f"{'─' * 55}")

    recall  = get_input("Word Recall Score          (0  – 20 )", 0,   20)
    seq     = get_input("Number Sequencing Time sec (30 – 120)", 30, 120)
    fluency = get_input("Speech Fluency words/min   (30 – 60 )", 30,  60)

    # Scale & Predict
    raw       = pd.DataFrame([[recall, seq, fluency]],
                             columns=["Word Recall Score",
                                      "Number Sequencing Time (sec)",
                                      "Speech Fluency (words/min)"])
    scaled    = scaler.transform(raw)
    pred      = model.predict(scaled)[0]
    proba     = model.predict_proba(scaled)[0]
    label_out = label_encoder.inverse_transform([pred])[0]
    confidence= proba.max() * 100

    # Risk colour indicator
    risk_icon = {"High Dementia Risk": "🔴", "Mild Cognitive Risk": "🟡", "No Cognitive Risk": "🟢"}
    icon = risk_icon.get(label_out, "⚪")

    print(f"\n  {'─'*53}")
    print(f"  PREDICTION RESULT — Patient {session}")
    print(f"  {'─'*53}")
    print(f"\n  {icon}  Predicted Risk Label : {label_out}")
    print(f"      Confidence           : {confidence:.2f}%")
    print(f"\n  Probability Breakdown:")
    for cls, prob in zip(label_encoder.classes_, proba):
        bar   = "█" * int(prob * 30)
        arrow = " ◀" if cls == label_out else ""
        print(f"    {cls:<26}: {prob:.4f}  {bar}{arrow}")

    print(f"\n  Input Summary:")
    print(f"    Word Recall Score         : {recall:.0f}")
    print(f"    Number Sequencing Time    : {seq:.0f} sec")
    print(f"    Speech Fluency            : {fluency:.0f} words/min")

    session += 1
    print(f"\n{'─' * 55}")
    again = input("  ➤  Predict another patient? (yes / no): ").strip().lower()
    if again not in ("yes", "y"):
        print("\n  ✅  Prediction session ended. Goodbye!\n")
        break

  DEMENTIA RISK PREDICTION SYSTEM
  Model : Random Forest (Best Model)

  Feature Input Ranges:
  ┌─────────────────────────────────────┬──────────┐
  │ Feature                             │  Range   │
  ├─────────────────────────────────────┼──────────┤
  │ Word Recall Score                   │  0 – 20  │
  │ Number Sequencing Time (seconds)    │ 30 – 120 │
  │ Speech Fluency (words/min)          │ 30 – 60  │
  └─────────────────────────────────────┴──────────┘


───────────────────────────────────────────────────────
  Patient 1 — Enter Feature Values
───────────────────────────────────────────────────────

  ─────────────────────────────────────────────────────
  PREDICTION RESULT — Patient 1
  ─────────────────────────────────────────────────────

  🟡  Predicted Risk Label : Mild Cognitive Risk
      Confidence           : 99.20%

  Probability Breakdown:
    High Dementia Risk        : 0.0028  
    Mild Cognitive Risk       : 0.9920  █████████████████████████████ ◀
    No Cognitiv